# CAP4770 Final Project – Feature Engineering & Preparation (Train/Val/Test Split, SMOTE)

## Objectives
- Split preprocessed data into training, validation, and test sets to prevent data leakage.
- Apply SMOTE to the training set to address class imbalance.
- Fit PCA on the training set and transform both train and test sets.

In [1]:
import pandas as pd
import numpy as np

# Load preprocessed data
X = pd.read_csv("../data/X_scaled.csv")
y = pd.read_csv("../data/y_binary.csv").squeeze()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

X shape: (2916697, 8)
y shape: (2916697,)

Class distribution:
is_ransomware
0    2875284
1      41413
Name: count, dtype: int64


### Train/Validation/Test Split

In [2]:
from sklearn.model_selection import train_test_split

# split train from rest of data
X_train, X_test_and_val, y_train, y_test_and_val = train_test_split(
    X, y, test_size=0.3, random_state=12, stratify=y
)

# split validation from test
X_val, X_test, y_val, y_test = train_test_split(
    X_test_and_val, y_test_and_val, test_size=0.5, random_state=12, stratify=y_test_and_val
)

print("Train size:", X_train.shape)
print("Val size:  ", X_val.shape)
print("Test size: ", X_test.shape)

Train size: (2041687, 8)
Val size:   (437505, 8)
Test size:  (437505, 8)


### SMOTE on Training Set

In [3]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=12)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Resampled train size:", X_train_smote.shape)
print("Resampled class distribution:")
print(pd.Series(y_train_smote).value_counts())

Resampled train size: (4025396, 8)
Resampled class distribution:
is_ransomware
0    2012698
1    2012698
Name: count, dtype: int64


### PCA on Training Set

In [4]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=12)
X_train_pca = pca.fit_transform(X_train_smote)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)

print(f"\nOriginal features:    {X_train_smote.shape[1]}")
print(f"PCA-reduced features: {X_train_pca.shape[1]}")
print(f"Variance explained:   {pca.explained_variance_ratio_.sum():.4f}")


Original features:    8
PCA-reduced features: 7
Variance explained:   0.9723


In [5]:
import os

# free up space if needed
os.remove("../data/X_scaled.csv")
os.remove("../data/y_binary.csv")

# save full feature splits
pd.DataFrame(X_train_smote).to_csv("../data/X_train.csv", index=False)
pd.DataFrame(X_val).to_csv("../data/X_val.csv", index=False)
pd.DataFrame(X_test).to_csv("../data/X_test.csv", index=False)
pd.Series(y_train_smote).to_csv("../data/y_train.csv", index=False)
pd.Series(y_val).to_csv("../data/y_val.csv", index=False)
pd.Series(y_test).to_csv("../data/y_test.csv", index=False)

# save PCA splits
pd.DataFrame(X_train_pca).to_csv("../data/X_train_pca.csv", index=False)
pd.DataFrame(X_val_pca).to_csv("../data/X_val_pca.csv", index=False)
pd.DataFrame(X_test_pca).to_csv("../data/X_test_pca.csv", index=False)

print("All splits saved to ../data/")

All splits saved to ../data/


## Next Steps

- Train baseline Decision Tree on full and PCA-reduced features
- Train Random Forest and Gradient Boosting on full and PCA-reduced features
- Evaluate each model using Precision, Recall, and F1-score